In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("data/notebook_dataset/cleaned_data.csv")

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Returned,TotalPrice,year,month,day,hour,day_of_week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,0,83.4,2009,12,1,7,1
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,0,81.0,2009,12,1,7,1
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,0,81.0,2009,12,1,7,1
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,0,100.8,2009,12,1,7,1
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,0,30.0,2009,12,1,7,1


In [4]:
df['Returned'].value_counts()

Returned
0    779495
1     18390
Name: count, dtype: int64

In [5]:
product_return_rate = df.groupby('StockCode')['Returned'].mean()

df['ProductReturnRate'] = df['StockCode'].map(product_return_rate)

In [6]:
df = pd.get_dummies(df, columns=['Country'], drop_first=True)

In [7]:
X = df.drop(columns=[
    'Returned',
    'Invoice',
    'Description',
    'Customer ID',
    'InvoiceDate',
    'StockCode'
])

y = df['Returned']

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
!pip install xgboost

In [10]:
X_train

,Quantity,Price,TotalPrice,year,month,day,hour,day_of_week,ProductReturnRate,Country_Austria,...,Country_Singapore,Country_Spain,Country_Sweden,Country_Switzerland,Country_Thailand,Country_USA,Country_United Arab Emirates,Country_United Kingdom,Country_Unspecified,Country_West Indies
153034,6,2.55,15.30,2010,5,18,10,1,0.046784,False,...,False,False,False,False,False,False,False,True,False,False
339149,2,2.95,5.90,2010,11,2,15,1,0.040816,False,...,False,False,False,False,False,False,False,True,False,False
655147,1,2.95,2.95,2011,9,23,14,4,0.044071,False,...,False,False,False,False,False,False,False,True,False,False
218340,1,7.49,7.49,2010,7,25,11,6,0.016216,False,...,False,False,False,False,False,False,False,True,False,False
339892,16,7.65,122.40,2010,11,3,10,2,0.056291,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
589183,24,0.39,9.36,2011,7,21,11,3,0.042857,False,...,False,False,False,False,False,False,False,True,False,False
683307,4,1.65,6.60,2011,10,11,12,1,0.006289,False,...,False,False,False,False,False,False,False,True,False,False
709694,12,2.10,25.20,2011,10,27,16,3,0.013245,False,...,False,False,False,False,False,False,False,True,False,False
223838,3,4.95,14.85,2010,7,29,14,3,0.018727,False,...,False,False,False,False,False,False,False,True,False,False


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


models = {
    
    "Logistic Regression": LogisticRegression(class_weight="balanced"),
    
    "Decision Tree": DecisionTreeClassifier(),
    
    "Random Forest": RandomForestClassifier(class_weight="balanced"),
    
    "Gradient Boosting": GradientBoostingClassifier(),
    
    "AdaBoost": AdaBoostClassifier(),
    
    "KNN": KNeighborsClassifier(),
    
    "Naive Bayes": GaussianNB(),
    
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    
    "LightGBM": LGBMClassifier()
    
}

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def evaluate_models(X_train, y_train, X_test, y_test, models):
    
    report = {}

    for name, model in models.items():
        
        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Probabilities for ROC-AUC
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:,1]
        else:
            y_prob = None

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        if y_prob is not None:
            roc_auc = roc_auc_score(y_test, y_prob)
        else:
            roc_auc = None

        report[name] = {
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        }

        print(f"\n{name} Results")
        print("Accuracy:", accuracy)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1 Score:", f1)
        print("ROC AUC:", roc_auc)

    return report

In [14]:
report = evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    models
)

c:\Users\chinm\anaconda3\envs\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression Results
Accuracy: 0.6232665108380281
Precision: 0.034031240711997625
Recall: 0.5603588907014682
F1 Score: 0.06416562889165629
ROC AUC: 0.6345339787394106

Decision Tree Results
Accuracy: 0.9643682986896608
Precision: 0.2581888246628131
Recall: 0.2914627514953779
F1 Score: 0.27381864623243934
ROC AUC: 0.6392738060591165

Random Forest Results
Accuracy: 0.9784806081076847
Precision: 0.6207920792079208
Recall: 0.17047308319738988
F1 Score: 0.2674914675767918
ROC AUC: 0.8918250745859948

Gradient Boosting Results
Accuracy: 0.9778852842201571
Precision: 0.7223880597014926
Recall: 0.06579662860250136
F1 Score: 0.12060802392225269
ROC AUC: 0.8464120192204444


c:\Users\chinm\anaconda3\envs\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



AdaBoost Results
Accuracy: 0.9769515657018242
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC AUC: 0.8137132308939956

KNN Results
Accuracy: 0.977697287203043
Precision: 0.5730061349693252
Recall: 0.12697117998912452
F1 Score: 0.2078789227687514
ROC AUC: 0.7143505546062591

Naive Bayes Results
Accuracy: 0.9408498718487
Precision: 0.09842464798550118
Recall: 0.19195214790647092
F1 Score: 0.13012625564464106
ROC AUC: 0.7238795520632753

XGBoost Results
Accuracy: 0.9799407182739367
Precision: 0.7738231917336394
Recall: 0.18325176726481784
F1 Score: 0.2963288634864805
ROC AUC: 0.8942007054936408
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 14712, number of negative: 623596
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036994 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total B

In [15]:
import pandas as pd

results_df = pd.DataFrame(report).T

results_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.623267,0.034031,0.560359,0.064166,0.634534
Decision Tree,0.964368,0.258189,0.291463,0.273819,0.639274
Random Forest,0.978481,0.620792,0.170473,0.267491,0.891825
Gradient Boosting,0.977885,0.722388,0.065797,0.120608,0.846412
AdaBoost,0.976952,0.000000,0.000000,0.000000,0.813713
KNN,0.977697,0.573006,0.126971,0.207879,0.714351
Naive Bayes,0.940850,0.098425,0.191952,0.130126,0.723880
XGBoost,0.979941,0.773823,0.183252,0.296329,0.894201
LightGBM,0.979176,0.752489,0.143828,0.241497,0.884777


In [16]:
pos = y_train.sum()
neg = len(y_train) - pos

scale_pos_weight = neg / pos

print(scale_pos_weight)

42.38689505165851


In [17]:

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))

print("ROC AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.99      0.85      0.92    155899
           1       0.11      0.77      0.19      3678

    accuracy                           0.85    159577
   macro avg       0.55      0.81      0.55    159577
weighted avg       0.97      0.85      0.90    159577

ROC AUC: 0.8939893648325966
